### ΛCDM

In [1]:
from getdist import plots , MCSamples , loadMCSamples
import numpy as np

Matplotlib is building the font cache; this may take a moment.


In [2]:
g=plots.get_subplot_plotter(chain_dir=[
                                       '/home/wangsy/chains_planck/base/plikHM_TTTEEE_lowl_lowE'
                                       ],
                                       width_inch=10)
roots = [ 'base_plikHM_TTTEEE_lowl_lowE']
params = ['omegabh2','omegach2','theta','tau','logA','ns','omegam','sigma8','H0']
g.triangle_plot(roots, params, filled=True, legend_labels=['TT+EE+TE+lowE+lowl'],title_limit=1)
g.export('planck_more.pdf')

/home/wangsy/chains_planck/base/plikHM_TTTEEE_lowl_lowE/base_plikHM_TTTEEE_lowl_lowE_3.txt
/home/wangsy/chains_planck/base/plikHM_TTTEEE_lowl_lowE/base_plikHM_TTTEEE_lowl_lowE_4.txt
/home/wangsy/chains_planck/base/plikHM_TTTEEE_lowl_lowE/base_plikHM_TTTEEE_lowl_lowE_1.txt
/home/wangsy/chains_planck/base/plikHM_TTTEEE_lowl_lowE/base_plikHM_TTTEEE_lowl_lowE_2.txt
Removed no burn in


In [3]:
from getdist import loadMCSamples
import numpy as np

readsamps = loadMCSamples('/home/wangsy/chains_planck/base/plikHM_TTTEEE_lowl_lowE/base_plikHM_TTTEEE_lowl_lowE')
cov = readsamps.cov(['omegabh2','omegach2','theta','tau','logA','ns','omegam','sigma8','S8','H0'])
mean = readsamps.mean(['omegabh2','omegach2','theta','tau','logA','ns','omegam','sigma8','S8','H0'])
fisher_planck = np.linalg.inv(cov)

In [4]:
mean

array([2.23597502e-02, 1.20200236e-01, 1.04090357e+00, 5.44450944e-02,
       3.04473522e+00, 9.64857442e-01, 3.16551317e-01, 8.11977605e-01,
       8.34045196e-01, 6.72733119e+01])

In [5]:
fisher_planck

array([[ 1.92435890e+11, -1.04891492e+11,  8.29653928e+10,
         2.06339312e+06, -2.16599914e+08, -1.64686954e+08,
         7.04935309e+09,  3.77970645e+08,  1.49955563e+08,
        -1.81392615e+08],
       [-1.04891492e+11,  6.15884691e+10, -4.22896826e+10,
        -7.28757505e+05,  6.73827426e+08,  5.09918726e+08,
        -4.85926004e+09, -1.80297763e+09,  1.39367453e+08,
         8.71262777e+07],
       [ 8.29653928e+10, -4.22896826e+10,  3.82426891e+10,
         9.07497654e+05,  3.40549404e+08,  2.56482177e+08,
         2.25991618e+09, -1.19052827e+09,  3.41510496e+08,
        -8.64386798e+07],
       [ 2.06339312e+06, -7.28757505e+05,  9.07497655e+05,
         1.54616173e+05, -5.74782828e+04, -4.24042500e+03,
        -1.16037917e+05, -1.78943785e+05,  1.45363926e+05,
        -1.69858173e+03],
       [-2.16599914e+08,  6.73827426e+08,  3.40549404e+08,
        -5.74782828e+04,  7.85565570e+07,  5.93035221e+07,
        -1.49265156e+08, -2.39319971e+08,  4.46288532e+07,
        -1.

In [6]:
fisher_rsd = np.array([[1278.81885787, 1839.07154863],
       [1839.07154863, 2987.87756339]])

fisher_rsd_ksz = np.array([[60014.72193115, 70818.55858603],
       [70818.55858603, 87513.25702333]])
# ["omegam", "sigma8"]

In [7]:
def add_matrix_elements(a, b):
    # 创建矩阵a的副本
    result = np.copy(a)
    result[6:8, 6:8] += b
    return result

In [8]:
planck_rsd_fish = add_matrix_elements(fisher_planck, fisher_rsd)
planck_rsd_ksz_fish = add_matrix_elements(fisher_planck, fisher_rsd_ksz)

planck_cov = np.linalg.inv(fisher_planck)
planck_rsd_cov = np.linalg.inv(planck_rsd_fish)
planck_rsd_ksz_cov = np.linalg.inv(planck_rsd_ksz_fish)

In [9]:
from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'\Omega_m', r'\sigma_8',r'S_8']
labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'\Omega_m', r'\sigma_8',r'S_8']
samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ tomography')


# Triangle plot
g = plots.get_subplot_plotter()
g.triangle_plot([samples0, samples,samples2], filled=True, width_inch=10)
g.export('fisher_total.pdf')

In [10]:
less0 = planck_cov[-3:,-3:]
less1 = planck_rsd_cov[-3:,-3:]
less2 = planck_rsd_ksz_cov[-3:,-3:]
mean1 = mean[-3:]

from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_m', r'\sigma_8',r'S_8']
labels =  [r'\Omega_m', r'\sigma_8',r'S_8']
samples0 = GaussianND(mean1, less0,names = names, labels = labels, label = 'Planck')
samples1 = GaussianND(mean1, less1,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean1,less2,names = names, labels = labels, label = 'Planck+RSD+kSZ tomography')


# Triangle plot
g = plots.get_subplot_plotter()
g.triangle_plot([samples0,samples1,samples2], filled=True, width_inch=7)
g.export('fisher_less.pdf')

In [12]:
less0

array([[7.09316477e-05, 2.66663782e-05, 1.20844955e-04],
       [2.66663782e-05, 5.37658615e-05, 9.03281642e-05],
       [1.20844955e-04, 9.03281642e-05, 2.51975638e-04]])

In [13]:
less1

array([[5.93736559e-05, 1.55170907e-05, 9.41698147e-05],
       [1.55170907e-05, 4.27402951e-05, 6.43186056e-05],
       [9.41698147e-05, 6.43186056e-05, 1.90126017e-04]])

In [11]:
less2

array([[ 2.36599418e-05, -1.33382268e-05,  1.74881846e-05],
       [-1.33382268e-05,  1.65793931e-05, -5.59393932e-07],
       [ 1.74881846e-05, -5.59393932e-07,  2.24819321e-05]])

In [2]:
import getdist
from getdist import plots
samples = getdist.loadMCSamples(r'/home/wangsy/chains_planck/base/plikHM_TTTEEE_lowl_lowE/base_plikHM_TTTEEE_lowl_lowE', settings={'ignore_rows':0.3})
p = samples.getParams()
samples.addDerived(p.omegabh2/(p.H0/100)**2, name='omegab', label=r'\Omega_b')
samples.addDerived(p.omegach2/(p.H0/100)**2, name='omegac', label=r'\Omega_c')
samples.addDerived(p.H0/100, name='h', label=r'h')
g = plots.get_single_plotter(width_inch=4)
mean1 = samples.mean(['omegab', 'omegac','h', 'omegal'])
mean1

array([0.04941387, 0.26571159, 0.67273312, 0.68344868])

### wCDM

In [2]:
from getdist import plots , MCSamples , loadMCSamples
import numpy as np

In [3]:
g=plots.get_subplot_plotter(chain_dir=['/home/wangsy/chains_planck/base_w/plikHM_TTTEEE_lowl_lowE',
                                       '/home/wangsy/chains_planck/base_w/plikHM_TTTEEE_lowl_lowE'
                                       ],
                                       width_inch=10)
roots = [ 'base_w_plikHM_TTTEEE_lowl_lowE','base_w_plikHM_TTTEEE_lowl_lowE_post_lensing']
params = ['omegabh2','omegach2','theta','tau','logA','ns','w','omegam','sigma8','S8']
g.triangle_plot(roots, params, filled=True, legend_labels=['TT+EE+TE+lowE+lowl','TT+EE+TE+lowE+lowl+post lensing'])
g.export('wcdm_planck.pdf')

In [4]:
### base_w_plikHM_TTTEEE_lowl_lowE
# Number of chains used =  4
#  var(mean)/mean(var), remaining chains, worst e-value: R-1 =       0.01268
# RL: Thin for Markov:  27
# RL: Thin for indep samples:   27
# RL: Estimated burn in steps:  156  ( 65  rows)
# using 24841 rows, 93 parameters; mean weight 2.3824322692323174, tot weight 59182.0
# Approx indep samples (N/corr length): 1813
# Equiv number of single samples (sum w)/max(w): 2192
# Effective number of weighted samples (sum w)^2/sum(w^2): 15133
# Best fit sample -log(Like) = 1383.562000
# mean(-Ln(like)) = 1394.278534
# -Ln(mean like)  = 1390.180784

In [5]:
### base_w_plikHM_TTTEEE_lowl_lowE_post_lensing
# Number of chains used =  4
#  var(mean)/mean(var), remaining chains, worst e-value: R-1 =       0.01838
# using 5849 rows, 94 parameters; mean weight 0.00869739933841236, tot weight 50.8710887303739
# Approx indep samples (N/corr length): 1394
# Equiv number of single samples (sum w)/max(w): 918
# Effective number of weighted samples (sum w)^2/sum(w^2): 4029
# Best fit sample -log(Like) = 1389.095000
# Ln(mean 1/like) = 1408.906801
# mean(-Ln(like)) = 1398.801956
# -Ln(mean like)  = 1394.684157

In [4]:
from getdist import loadMCSamples
import numpy as np

readsamps = loadMCSamples('/home/wangsy/chains_planck/base_w/plikHM_TTTEEE_lowl_lowE/base_w_plikHM_TTTEEE_lowl_lowE')
cov = readsamps.cov( ['omegabh2','omegach2','theta','tau','logA','ns','w','omegam','sigma8','S8','omegal'])
mean = readsamps.mean( ['omegabh2','omegach2','theta','tau','logA','ns','w','omegam','sigma8','S8','omegal'])
fisher_planck = np.linalg.inv(cov)

In [ ]:
#fisher_planck

In [11]:
mean

array([ 0.02239316,  0.11992714,  1.04093817,  0.05393335,  3.04338637,
        0.96539223, -1.58491121,  0.19765297,  0.97234554,  0.77748323,
        0.80234703])

In [5]:
# ['w','omegam','sigma8']  [6 7 8]--[0 1 2]
fisher_rsd_ksz = np.array([[  1124.24815622, -11659.46790013,  -8182.72112514],
       [-11659.46790013, 143252.53907746,  87984.41764765],
       [ -8182.72112514,  87984.41764765,  61573.54789072]])
fisher_rsd = np.array([[  30.20200579, -192.07318699, -114.12502183],
       [-192.07318699, 2024.64107128,  858.61920149],
       [-114.12502183,  858.61920149,  496.05759245]])

In [6]:
def add_fisher(a,b):
    result = np.copy(a)
    result[6:9,6:9] += b
    return result

In [7]:
planck_rsd_fish = add_fisher(fisher_planck, fisher_rsd)
planck_rsd_ksz_fish = add_fisher(fisher_planck, fisher_rsd_ksz)

planck_rsd_cov = np.linalg.inv(planck_rsd_fish)
planck_rsd_ksz_cov = np.linalg.inv(planck_rsd_ksz_fish)

In [ ]:
from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w',r'\Omega_m', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w',r'\Omega_m', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ tomography')


# Triangle plot
g = plots.get_subplot_plotter()
g.triangle_plot([samples0,samples,samples2], filled=True, width_inch=10)
#g.export('wcdm2.pdf')

In [7]:
from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND
def rgba(cc):
    return tuple(cc/255.)
c1 = rgba(np.array([41.,157.,143.,255.]))
c2 = rgba(np.array([233.,196.,106.,255.]))
c3 = rgba(np.array([216.,118.,89.,255.]))

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w',r'\Omega_m', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w',r'\Omega_m', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ')

g = plots.get_subplot_plotter()
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add=0.8
g.settings.title_limit_fontsize = 14
g.triangle_plot([samples0, samples, samples2], ['w','\Omega_\Lambda', '\\sigma_8','S_8'], 
    filled=True, 
    contour_lws=1.5,
    legend_labels=['Base', 'Base+RSD', 'Base+RSD+kSZ'], 
    legend_loc='upper right',
    contour_colors=[c2,c1,c3],
    markers={'w':-1.584, '\Omega_\Lambda':0.802, '\\sigma_8':0.972,'S_8':0.777})#-1.58491121, 0.80234703,  0.97234554,  0.77748323

for ax in g.subplots[:, :].flat:
    if ax is not None:  # 检查 ax 是否为 None
        ax.minorticks_on()
        ax.tick_params(which='both', direction='in', top=True, right=True)
        ax.tick_params(which='minor', length=4)
        ax.tick_params(which='major', length=7)
        
g.export('wcdm_use.pdf')

# Triangle plot


In [18]:
from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND
def rgba(cc):
    return tuple(cc/255.)
c1 = rgba(np.array([41.,157.,143.,255.]))
c2 = rgba(np.array([233.,196.,106.,255.]))
c3 = rgba(np.array([216.,118.,89.,255.]))

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w',r'\Omega_m', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w',r'\Omega_m', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ')

g = plots.get_single_plotter(width_inch=4, ratio=1)
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add=0.8
g.settings.title_limit_fontsize = 14
g.plot_2d([samples0, samples, samples2], ['w', '\\sigma_8'],filled=True, colors=[c2,c1,c3],lims=[-2.5, -0.7, 0.75, 1.2]) 
#g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'], colored_text=True)
g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'], legend_loc='upper right');
    
g.add_x_marker(-1.584)
g.add_y_marker(0.972)

for ax in g.subplots[:, :].flat:
    if ax is not None:  # 检查 ax 是否为 None
        ax.minorticks_on()
        ax.tick_params(which='both', direction='in', top=True, right=True)
        ax.tick_params(which='minor', length=4)
        ax.tick_params(which='major', length=7)
        
g.export('wcdm_use_ppt.pdf')

### CPL

In [19]:
from getdist import plots , MCSamples , loadMCSamples
import numpy as np

In [30]:
g=plots.get_subplot_plotter(chain_dir=['/home/wangsy/chains_planck/base_w_wa/plikHM_TTTEEE_lowl_lowE_BAO',
                                       '/home/wangsy/chains_planck/base_w_wa/plikHM_TTTEEE_lowl_lowE_BAO'
                                       ],
                                       width_inch=10)
roots = [ 'base_w_wa_plikHM_TTTEEE_lowl_lowE_BAO','base_w_wa_plikHM_TTTEEE_lowl_lowE_BAO_post_lensing']
params = ['omegabh2','omegach2','theta','tau','logA','ns','w0','wa','omegam','sigma8','S8','omegal']
g.triangle_plot(roots, params, filled=True, legend_labels=['TT+EE+TE+lowE+lowl+BAO','TT+EE+TE+lowE+lowl+BAO+post_lensing'])
g.export('cpl_planck.pdf')

In [44]:
### base_w_wa_plikHM_TTTEEE_lowl_lowE_BAO
# Number of chains used =  4
#  var(mean)/mean(var), remaining chains, worst e-value: R-1 =       0.01370
# RL: Thin for Markov:  29
# RL: Thin for indep samples:   29
# RL: Estimated burn in steps:  126  ( 55  rows)
# using 31258 rows, 98 parameters; mean weight 2.298515580011517, tot weight 71847.0
# Approx indep samples (N/corr length): 1941
# Equiv number of single samples (sum w)/max(w): 2477
# Effective number of weighted samples (sum w)^2/sum(w^2): 19588
# Best fit sample -log(Like) = 1387.439000
# mean(-Ln(like)) = 1398.453899
# -Ln(mean like)  = 1394.179580

In [45]:
### base_w_wa_plikHM_TTTEEE_lowl_lowE_BAO_post_lensing
# Removed 0.3 as burn in
# Number of chains used =  4
#  var(mean)/mean(var), remaining chains, worst e-value: R-1 =       0.01809
# using 7132 rows, 99 parameters; mean weight 0.008367313654728575, tot weight 59.675680985524195
# Approx indep samples (N/corr length): 1678
# Equiv number of single samples (sum w)/max(w): 1632
# Effective number of weighted samples (sum w)^2/sum(w^2): 5698
# Best fit sample -log(Like) = 1393.800000
# mean(-Ln(like)) = 1402.919786
# -Ln(mean like)  = 1398.710926

In [20]:
from getdist import loadMCSamples
import numpy as np

readsamps = loadMCSamples('/home/wangsy/chains_planck/base_w_wa/plikHM_TTTEEE_lowl_lowE_BAO/base_w_wa_plikHM_TTTEEE_lowl_lowE_BAO')
cov = readsamps.cov(['omegabh2','omegach2','theta','tau','logA','ns','w0','wa','sigma8','S8','omegal'])
mean = readsamps.mean(['omegabh2','omegach2','theta','tau','logA','ns','w0','wa','sigma8','S8','omegal'])
fisher_planck = np.linalg.inv(cov)

In [3]:
mean

array([ 0.02235572,  0.12027999,  1.04089311,  0.05383925,  3.04377334,
        0.96454503, -0.57779661, -1.31630327,  0.79547449,  0.8475389 ,
        0.65833267])

In [21]:

# ['w0','wa','sigma8']
fisher_rsd_ksz = np.array([[ 3.06950329e+02,  4.54869568e+01, -3.22024254e+03],
       [ 4.54869568e+01,  7.82400579e+00, -4.68728658e+02],
       [-3.22024254e+03, -4.68728658e+02,  3.51834507e+04]])
fisher_rsd = np.array([[ 1.71114915e+01,  2.52397287e+00, -1.41788328e+02],
       [ 2.52397287e+00,  1.15741656e+00, -2.00687738e+01],
       [-1.41788328e+02, -2.00687738e+01,  1.35138449e+03]])


In [22]:

def add_fisher(a,b):
    result = np.copy(a)
    result[6:9,6:9] += b
    return result

In [23]:

planck_rsd_fish = add_fisher(fisher_planck, fisher_rsd)
planck_rsd_ksz_fish = add_fisher(fisher_planck, fisher_rsd_ksz)

planck_rsd_cov = np.linalg.inv(planck_rsd_fish)
planck_rsd_ksz_cov = np.linalg.inv(planck_rsd_ksz_fish)

In [10]:
#from getdist import plots, MCSamples
#from getdist.gaussian_mixtures import GaussianND
#
## Get the getdist MCSamples objects for the samples, specifying same parameter
## names and labels; if not specified weights are assumed to all be unity
#names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'\Omega_m', r'\sigma_8',r'w_0',r'w_a',r'S_8']
#labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'\Omega_m', r'\sigma_8',r'w_0',r'w_a',r'S_8']
#samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
#samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
#samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ tomography')
#
#
## Triangle plot
#g = plots.get_subplot_plotter()
#g.triangle_plot([samples0,samples,samples2], filled=True, width_inch=10)
#g.export('cpl2.pdf')

In [13]:

from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND
def rgba(cc):
    return tuple(cc/255.)
c1 = rgba(np.array([41.,157.,143.,255.]))
c2 = rgba(np.array([233.,196.,106.,255.]))
c3 = rgba(np.array([216.,118.,89.,255.]))

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w_0',r'w_a', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w_0',r'w_a', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ')


g = plots.get_subplot_plotter()
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add=0.8
g.settings.title_limit_fontsize = 14
g.triangle_plot([samples0, samples, samples2], ['w_0','w_a','\\Omega_\Lambda','\\sigma_8','S_8'], 
    filled=True, 
    contour_lws=1.5,
    legend_labels=['Base', 'Base+RSD', 'Base+RSD+kSZ'], 
    legend_loc='upper right',
    contour_colors=[c2,c1,c3],
    markers={'w_0':-0.577, 'w_a':-1.316,'\\sigma_8':0.795, 'S_8':0.847, '\Omega_\Lambda': 0.658}
    #-0.57779661, -1.31630327,  0.79547449,  0.8475389 , 0.65833267
    )

for ax in g.subplots[:, :].flat:
    if ax is not None:  # 检查 ax 是否为 None
        ax.minorticks_on()
        ax.tick_params(which='both', direction='in', top=True, right=True)
        ax.tick_params(which='minor', length=4)
        ax.tick_params(which='major', length=7)
            
g.export('cpl_use.pdf')

In [29]:
from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND
def rgba(cc):
    return tuple(cc/255.)
c1 = rgba(np.array([41.,157.,143.,255.]))
c2 = rgba(np.array([233.,196.,106.,255.]))
c3 = rgba(np.array([216.,118.,89.,255.]))

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w_0',r'w_a', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w_0',r'w_a', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ')


#g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'], colored_text=True)
#g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'], legend_loc='upper right');

g = plots.get_single_plotter(width_inch=4, ratio=1)
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add=0.8
g.settings.title_limit_fontsize = 14
g.plot_2d([samples0, samples, samples2], ['w_0','\\sigma_8'], filled=True, colors=[c2,c1,c3],lims=[-1.5, 0.5, 0.7, 0.9])
g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'],legend_loc='upper right');
g.add_x_marker(-0.577)
g.add_y_marker(0.795)
#    markers={'w_0':-0.577, 'w_a':-1.316,'\\sigma_8':0.795, 'S_8':0.847, '\Omega_\Lambda': 0.658}
    #-0.57779661, -1.31630327,  0.79547449,  0.8475389 , 0.65833267


for ax in g.subplots[:, :].flat:
    if ax is not None:  # 检查 ax 是否为 None
        ax.minorticks_on()
        ax.tick_params(which='both', direction='in', top=True, right=True)
        ax.tick_params(which='minor', length=4)
        ax.tick_params(which='major', length=7)
            
g.export('cpl_use_ppt1.pdf')

In [32]:
from getdist import plots, MCSamples
from getdist.gaussian_mixtures import GaussianND
def rgba(cc):
    return tuple(cc/255.)
c1 = rgba(np.array([41.,157.,143.,255.]))
c2 = rgba(np.array([233.,196.,106.,255.]))
c3 = rgba(np.array([216.,118.,89.,255.]))

# Get the getdist MCSamples objects for the samples, specifying same parameter
# names and labels; if not specified weights are assumed to all be unity
names = [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w_0',r'w_a', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
labels =  [r'\Omega_b h^2',r'\Omega_c h^2',r'100\theta_{MC}',r'\tau',r'{\rm{ln}}(10^{10} A_s)',r'n_s',r'w_0',r'w_a', r'\sigma_8',r'S_8',r'\Omega_\Lambda']
samples0 = GaussianND(mean, cov,names = names, labels = labels, label = 'Planck')
samples = GaussianND(mean, planck_rsd_cov,names = names, labels = labels, label = 'Planck+RSD')
samples2 = GaussianND(mean,planck_rsd_ksz_cov,names = names, labels = labels, label = 'Planck+RSD+kSZ')


#g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'], colored_text=True)
#g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'], legend_loc='upper right');

g = plots.get_single_plotter(width_inch=4, ratio=1)
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add=0.8
g.settings.title_limit_fontsize = 14
g.plot_2d([samples0, samples, samples2], ['w_a','\\sigma_8'], filled=True, colors=[c2,c1,c3],lims=[-4, 1, 0.7, 0.9])
g.add_legend(['Base', 'Base+RSD', 'Base+RSD+kSZ'],legend_loc='upper right');
g.add_x_marker(-1.316)
g.add_y_marker(0.795)
#    markers={'w_0':-0.577, 'w_a':-1.316,'\\sigma_8':0.795, 'S_8':0.847, '\Omega_\Lambda': 0.658}
    #-0.57779661, -1.31630327,  0.79547449,  0.8475389 , 0.65833267


for ax in g.subplots[:, :].flat:
    if ax is not None:  # 检查 ax 是否为 None
        ax.minorticks_on()
        ax.tick_params(which='both', direction='in', top=True, right=True)
        ax.tick_params(which='minor', length=4)
        ax.tick_params(which='major', length=7)
            
g.export('cpl_use_ppt2.pdf')

### 中微子

In [1]:
from getdist import plots , MCSamples , loadMCSamples
import numpy as np

In [5]:
g=plots.get_subplot_plotter(chain_dir=['/home/wangsy/chains_planck/base_mnu/plikHM_TTTEEE_lowl_lowE'],
                                       width_inch=10)
roots = [ 'base_mnu_plikHM_TTTEEE_lowl_lowE']
params = ['theta','tau','logA','ns','omegabh2','omegach2','mnu','H0','omegam','sigma8']
g.triangle_plot(roots, params, filled=True, legend_labels=['TT+EE+TE+lowE+lowl'],title_limit=1)
g.export('neutrino_planck.pdf')

In [7]:
from getdist import loadMCSamples
import numpy as np

readsamps = loadMCSamples('/home/wangsy/chains_planck/base_mnu/plikHM_TTTEEE_lowl_lowE/base_mnu_plikHM_TTTEEE_lowl_lowE')
cov = readsamps.cov(['theta','tau','logA','ns','omegabh2','omegach2','mnu','H0','omegam','sigma8'])
mean = readsamps.mean(['theta','tau','logA','ns','omegabh2','omegach2','mnu','H0','omegam','sigma8'])
fisher_planck = np.linalg.inv(cov)

In [8]:
mean

array([1.04089478e+00, 5.45000477e-02, 3.04520429e+00, 9.64622533e-01,
       2.23496499e-02, 1.20245632e-01, 9.08195042e-02, 6.70303690e+01,
       3.19907224e-01, 8.06897740e-01])